# 04 — Prédire des métadonnées depuis les représentations

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Question étudiée

Le type de chauffage réellement disponible (`CHAUFFAGE`) est-il associé aux profils ? Chaque foyer devient **une seule observation**. Nous comparons la même régression logistique pondérée sur variables métier, ACP, latent, et combinaison :

$$P(Y=k\mid x)=\frac{\exp(\beta_k^\top x)}{\sum_j\exp(\beta_j^\top x)}.$$

Les partitions sont faites par foyer ; encodeur, PCA et normalisation sont ajustés sur les foyers d'apprentissage.

In [ ]:
pred=pd.DataFrame(results['metadata_prediction']).T
display(pred[['balanced_accuracy','macro_f1','majority_balanced_accuracy','n_train_clients','n_test_clients']])
pred[['balanced_accuracy','macro_f1','majority_balanced_accuracy']].plot.bar(figsize=(10,3)); plt.axhline(1/3,color='k',ls='--',lw=1); plt.ylabel('Score test'); plt.show()

In [ ]:
chosen=results['metadata_prediction']['pca']
cm=pd.DataFrame(chosen['confusion_matrix'],index=chosen['classes'],columns=chosen['classes'])
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues'); plt.xlabel('Prédit'); plt.ylabel('Réel'); plt.title('Matrice de confusion — représentation ACP'); plt.show()
display(pd.Series({k:v.get('roc_auc_ovr_macro') for k,v in results['metadata_prediction'].items()},name='ROC-AUC OVR macro'))

In [ ]:
import pyarrow.parquet as pq
meta_source=pd.read_parquet(ROOT/'data/df_meta_gradient.parquet',columns=['ID','CHAUFFAGE'])
display(meta_source.CHAUFFAGE.value_counts(dropna=False).to_frame('effectif complet'))

## Interprétation et limites

Le test ne contient que huit foyers en mode smoke. Les scores sous la référence majoritaire sont donc un contrôle de plomberie, pas une conclusion substantielle. Le jeu complet est fortement déséquilibré (`Autre` domine) ; balanced accuracy et macro-F1 sont plus informatives que l'accuracy brute. Une ROC-AUC multiclasses ne sera ajoutée qu'avec assez d'exemples par classe.

Questions : quelles variables métier risquent de fuiter la cible ? Pourquoi une journée étiquetée par son foyer n'est-elle pas une observation indépendante ? Comment fixer un test stable pour une courbe d'apprentissage en nombre de foyers étiquetés ?